# HAAQI_Spoof — Training on SceneFake Dataset

**Model:** `HAAQI_Spoof` (Wav2Vec2 + BLSTM + Attention)  
**Dataset:** [SceneFake on Kaggle](https://www.kaggle.com/datasets/mohammedabdeldayem/scenefake)  
**Label convention:** `0 = real (bonafide)`, `1 = fake (spoof)`  
**Output:** Sigmoid score → BCELoss  
**Saved model:** `haaqi_spoof_scenefake.pth`

## Step 1 — Install dependencies

In [1]:
!pip install -q transformers torchaudio kaggle

## Step 2 — Enter Kaggle API credentials and download the dataset

Run this cell. It will ask for your **Kaggle username** and **API key**.  
Find these at: [kaggle.com](https://www.kaggle.com) → Your Profile → Settings → API → **Create New Token**  
Your username and key are inside the `kaggle.json` that gets downloaded.

In [2]:
import os
import json
from getpass import getpass

# Prompt for credentials (getpass hides the key as you type)
kaggle_username = input('Enter your Kaggle username: ').strip()
kaggle_key      = getpass('Enter your Kaggle API key: ').strip()

# Write kaggle.json
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': kaggle_username, 'key': kaggle_key}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

print('Credentials saved. Downloading dataset...')

# Download SceneFake dataset
!kaggle datasets download -d mohammedabdeldayem/scenefake -p /content/scenefake --unzip

print('\nDataset downloaded. Folder structure:')
!find /content/scenefake -maxdepth 3 -type d

Enter your Kaggle username: atharvapandey27
Enter your Kaggle API key: ··········
Credentials saved. Downloading dataset...
Dataset URL: https://www.kaggle.com/datasets/mohammedabdeldayem/scenefake
License(s): CC-BY-NC-SA-4.0
100% 5.37G/5.37G [04:10<00:00, 23.0MB/s]


Dataset downloaded. Folder structure:
/content/scenefake
/content/scenefake/train
/content/scenefake/train/real
/content/scenefake/train/fake
/content/scenefake/eval
/content/scenefake/eval/real
/content/scenefake/eval/fake
/content/scenefake/dev
/content/scenefake/dev/real
/content/scenefake/dev/fake


## Step 3 — Set dataset root path

After the cell above prints the folder structure, confirm the path below is correct.  
It should point to the folder that contains `train/`, `dev/`, `eval/`.

In [3]:
DATABASE_PATH = '/content/scenefake'  # adjust if the unzip created a subfolder

## Step 4 — Upload haaqi_model.py

In [4]:
from google.colab import files
files.upload()  # upload haaqi_model.py

Saving haaqi_model.py to haaqi_model.py


{'haaqi_model.py': b'import torch\r\nimport torch.nn as nn\r\nfrom transformers import Wav2Vec2Model\r\n\r\nclass BLSTM(nn.Module):\r\n    def __init__(self, input_size=768, hidden_size=128):\r\n        super().__init__()\r\n        self.lstm = nn.LSTM(input_size, hidden_size,\r\n                            batch_first=True, bidirectional=True)\r\n\r\n    def forward(self, x):\r\n        out, _ = self.lstm(x)\r\n        return out\r\n\r\nclass Attention(nn.Module):\r\n    def __init__(self, hidden_size=256):\r\n        super().__init__()\r\n        self.attn = nn.Linear(hidden_size, 1)\r\n\r\n    def forward(self, x):\r\n        weights = torch.softmax(self.attn(x), dim=1)\r\n        context = torch.sum(weights * x, dim=1)\r\n        return context\r\n\r\nclass HAAQI_Spoof(nn.Module):\r\n    def __init__(self):\r\n        super().__init__()\r\n        self.wav2vec = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h")\r\n        self.blstm = BLSTM()\r\n        self.attn = Atten

In [5]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

## Step 5 — Imports

In [6]:
import os
import torch
import torchaudio
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from torch.optim import Adam
from haaqi_model import HAAQI_Spoof

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

Using device: cuda


## Step 6 — Dataset class

In [7]:
TARGET_SR = 16000
MAX_SAMPLES = TARGET_SR * 4  # 4 seconds of audio

class SceneFakeDataset(Dataset):
    """
    Reads .wav files from:
        database_path/split/real/*.wav  → label 0
        database_path/split/fake/*.wav  → label 1

    Resamples to 16 kHz, converts to mono, pads/trims to MAX_SAMPLES.
    """
    def __init__(self, database_path, split):
        self.samples = []  # list of (filepath, label)
        for label_name, label in [('real', 0), ('fake', 1)]:
            folder = os.path.join(database_path, split, label_name)
            if not os.path.isdir(folder):
                raise FileNotFoundError(f'Expected folder not found: {folder}')
            for fname in os.listdir(folder):
                if fname.endswith('.wav'):
                    self.samples.append((os.path.join(folder, fname), label))
        print(f'[{split}] Loaded {len(self.samples)} files')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        waveform, sr = torchaudio.load(path)

        # Convert to mono
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        # Resample if needed
        if sr != TARGET_SR:
            waveform = T.Resample(sr, TARGET_SR)(waveform)

        # Pad or trim to MAX_SAMPLES
        n = waveform.shape[1]
        if n < MAX_SAMPLES:
            waveform = torch.nn.functional.pad(waveform, (0, MAX_SAMPLES - n))
        else:
            waveform = waveform[:, :MAX_SAMPLES]

        # HAAQI_Spoof expects shape (batch, samples) — remove channel dim
        waveform = waveform.squeeze(0)  # (MAX_SAMPLES,)

        return waveform, torch.tensor(label, dtype=torch.float32)

## Step 7 — Hyperparameters

In [8]:
BATCH_SIZE = 8       # reduce to 4 if you get OOM errors
NUM_EPOCHS = 10
LR         = 1e-4
SAVE_PATH  = 'haaqi_spoof_scenefake.pth'

## Step 8 — Build dataloaders

In [9]:
train_dataset = SceneFakeDataset(DATABASE_PATH, 'train')
dev_dataset   = SceneFakeDataset(DATABASE_PATH, 'dev')
eval_dataset  = SceneFakeDataset(DATABASE_PATH, 'eval')

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
dev_loader   = DataLoader(dev_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
eval_loader  = DataLoader(eval_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

[train] Loaded 13185 files
[dev] Loaded 12843 files
[eval] Loaded 32746 files


In [10]:
x = torch.randn(2,3).cuda()
print(x)

tensor([[ 0.7622, -1.1962, -0.1421],
        [ 0.7515,  0.3291, -0.6309]], device='cuda:0')


## Step 9 — Model, optimizer, loss

In [11]:
model     = HAAQI_Spoof().to(DEVICE)
optimizer = Adam(model.parameters(), lr=LR)
criterion = nn.BCEWithLogitsLoss()  # model outputs sigmoid → BCELoss

print(model)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


HAAQI_Spoof(
  (wav2vec): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2GroupNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
          (activation): GELUActivation()
          (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
        )
        (1-4): 4 x Wav2Vec2NoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2NoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projection): Linear(in_features=512, out_features=768, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): Wa

## Step 10 — Training loop

In [12]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for waveforms, labels in loader:
            waveforms = waveforms.to(DEVICE)   # (B, samples)
            labels    = labels.float().to(DEVICE)       # (B,)
            preds     = model(waveforms).squeeze(1)  # (B,)
            loss      = criterion(preds, labels)
            total_loss += loss.item() * len(labels)
            predicted = (preds >= 0.5).float()
            correct  += (predicted == labels).sum().item()
            total    += len(labels)
    return total_loss / total, correct / total


best_dev_loss = float('inf')

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running_loss, running_correct, running_total = 0.0, 0, 0

    for batch_idx, (waveforms, labels) in enumerate(train_loader):
        waveforms = waveforms.to(DEVICE)   # (B, samples)
        labels    = labels.to(DEVICE)       # (B,)

        optimizer.zero_grad()
        preds = model(waveforms).squeeze(1)  # (B,)
        loss  = criterion(preds, labels)
        loss.backward()
        optimizer.step()

        running_loss    += loss.item() * len(labels)
        predicted        = (preds >= 0.5).float()
        running_correct += (predicted == labels).sum().item()
        running_total   += len(labels)

        if (batch_idx + 1) % 50 == 0:
            print(f'  Epoch {epoch} | Batch {batch_idx+1}/{len(train_loader)} '
                  f'| Loss: {running_loss/running_total:.4f} '
                  f'| Acc: {running_correct/running_total:.4f}')

    train_loss = running_loss / running_total
    train_acc  = running_correct / running_total

    dev_loss, dev_acc = evaluate(model, dev_loader, criterion)

    print(f'\nEpoch {epoch}/{NUM_EPOCHS} Summary:')
    print(f'  Train  — Loss: {train_loss:.4f} | Acc: {train_acc:.4f}')
    print(f'  Dev    — Loss: {dev_loss:.4f}   | Acc: {dev_acc:.4f}')

    # Save best model based on dev loss
    if dev_loss < best_dev_loss:
        best_dev_loss = dev_loss
        torch.save(model.state_dict(), SAVE_PATH)
        print(f'  ✓ Best model saved to {SAVE_PATH}\n')
    else:
        print()

  Epoch 1 | Batch 50/1649 | Loss: 0.5562 | Acc: 0.7650
  Epoch 1 | Batch 100/1649 | Loss: 0.5271 | Acc: 0.7987
  Epoch 1 | Batch 150/1649 | Loss: 0.5126 | Acc: 0.8100
  Epoch 1 | Batch 200/1649 | Loss: 0.5092 | Acc: 0.8113
  Epoch 1 | Batch 250/1649 | Loss: 0.5036 | Acc: 0.8155
  Epoch 1 | Batch 300/1649 | Loss: 0.5007 | Acc: 0.8175
  Epoch 1 | Batch 350/1649 | Loss: 0.5011 | Acc: 0.8164
  Epoch 1 | Batch 400/1649 | Loss: 0.5014 | Acc: 0.8156
  Epoch 1 | Batch 450/1649 | Loss: 0.5008 | Acc: 0.8158
  Epoch 1 | Batch 500/1649 | Loss: 0.5033 | Acc: 0.8130
  Epoch 1 | Batch 550/1649 | Loss: 0.5033 | Acc: 0.8127
  Epoch 1 | Batch 600/1649 | Loss: 0.5039 | Acc: 0.8119
  Epoch 1 | Batch 650/1649 | Loss: 0.5085 | Acc: 0.8071
  Epoch 1 | Batch 700/1649 | Loss: 0.5087 | Acc: 0.8068
  Epoch 1 | Batch 750/1649 | Loss: 0.5073 | Acc: 0.8080
  Epoch 1 | Batch 800/1649 | Loss: 0.5081 | Acc: 0.8070
  Epoch 1 | Batch 850/1649 | Loss: 0.5076 | Acc: 0.8075
  Epoch 1 | Batch 900/1649 | Loss: 0.5072 | Acc: 

## Step 11 — Evaluate on eval set using best saved model

In [13]:
import numpy as np
from sklearn.metrics import accuracy_score, roc_curve
from tqdm import tqdm

!pip install -q scikit-learn tqdm

def compute_eer(y_true, y_scores):
    fpr, tpr, _ = roc_curve(y_true, y_scores)
    fnr = 1 - tpr
    idx = np.nanargmin(np.abs(fnr - fpr))
    return fpr[idx]

# Load best model
model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
model.eval()

all_preds, all_labels, all_scores = [], [], []

eval_bar = tqdm(eval_loader, desc='Evaluating', ncols=80)

with torch.no_grad():
    for x, y in eval_bar:
        x = x.to(DEVICE)

        # shape fix matching evaluate_haaqi.py
        if len(x.shape) == 3:
            x = x.squeeze(1)

        output = model(x).squeeze()

        scores = output.cpu().numpy()
        preds  = (scores > 0.5).astype(int)

        all_scores.extend(scores)
        all_preds.extend(preds)
        all_labels.extend(y.numpy())

accuracy = accuracy_score(all_labels, all_preds) * 100
eer      = compute_eer(all_labels, all_scores)

print('=' * 50)
print('HAAQI MODEL RESULTS')
print('=' * 50)
print(f'Total Samples : {len(all_labels)}')
print(f'Accuracy      : {accuracy:.2f}%')
print(f'EER           : {eer:.4f}')
print('=' * 50)

import os
os.makedirs('outputs', exist_ok=True)
with open('outputs/results_haaqi.txt', 'w') as f:
    f.write('HAAQI MODEL RESULTS\n')
    f.write('='*50 + '\n')
    f.write(f'Accuracy: {accuracy:.2f}%\n')
    f.write(f'EER: {eer:.4f}\n')

print('Results saved to outputs/results_haaqi.txt')

Evaluating: 100%|███████████████████████████| 4094/4094 [13:00<00:00,  5.24it/s]


HAAQI MODEL RESULTS
Total Samples : 32746
Accuracy      : 80.66%
EER           : 0.0000
Results saved to outputs/results_haaqi.txt


## Step 12 — Download the saved .pth file to your machine

In [14]:
from google.colab import files
files.download(SAVE_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>